# Local Context Store Setup
Validate developer context and build the local SQLite/FTS5 retrieval index.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from dq_agent.config import config_summary, load_app_config
from dq_agent.context_store import make_context_retriever
from dq_agent.context_utils import configure_workflow_logging, context_workflow_paths, logged_step

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
config = load_app_config(ROOT)
paths = context_workflow_paths(config, RUN_ID)
logger = configure_workflow_logging(paths['log'], config.project.log_level)
print(config_summary(config))

In [ ]:
with logged_step(logger, paths['checkpoint'], 'REBUILD_LOCAL_CONTEXT'):
    store = make_context_retriever(config, logger)
    result = store.rebuild()
    print({'database': str(store.path), **result})

In [ ]:
with logged_step(logger, paths['checkpoint'], 'VERIFY_CONTEXT_RETRIEVAL'):
    sample = store.search({'description': 'sales measure relationship active current filter'}, limit=8)
    display([{k: row.get(k) for k in ('record_id','context_type','subject_key','retrieval_score','origin')} for row in sample])

The database is generated from versioned YAML and can be deleted and rebuilt. Developer context is immediately trusted; learned context enters only through approval processing.